# 聖尼古拉斯與三姐妹：在 Colab 渲染（油畫厚塗版）

這本筆記會把 [FelixF048/Santa](https://github.com/FelixF048/Santa) 的短片程式逐格渲染成 5 分鐘、1080p 的 mp4。

- 畫面是程式碼畫的（WebGL2 厚塗引擎），不使用影片生成模型。
- **請先選 T4 GPU 執行階段**（執行階段 → 變更執行階段類型；有「大量 RAM」更好）。沒有 GPU 也跑得動，但全片要數小時。
- 依序執行每一格即可。第 2 格是整條流程，T4 上約 6–8 分鐘。

## 1. 取得程式

In [ ]:
BRANCH = 'impasto'   # 厚塗版所在的分支；合併進 main 之後改成 'main'
!cd /content && rm -rf Santa && git clone -q -b {BRANCH} https://github.com/FelixF048/Santa.git Santa
!cd /content/Santa && git log --oneline -1 && (nvidia-smi --query-gpu=name --format=csv,noheader || echo "沒有 GPU")

## 2. 一鍵流程：安裝 → 渲染 → 配樂 → 配音 → 驗證
`run_all.sh` 會補裝 NVIDIA 的繪圖函式庫，試出瀏覽器能用的 GPU 繪圖方式，再渲染全片。

**注意**：配音這一步會把字幕文字送到微軟的線上語音服務（edge-tts）。不想外送文字，就改成只跑 `node render/render.mjs`，再自己配音。

In [ ]:
!cd /content/Santa && bash colab/run_all.sh 2>&1 | tee /content/run.log

## 3. 看抽格圖（每 12.5 秒一格）

In [ ]:
from IPython.display import Image, display
display(Image('/content/Santa/out/check/sheet.jpg', width=900))

## 4. 存到 Google 雲端硬碟（會跳出授權視窗）

In [ ]:
from google.colab import drive
import shutil, os
drive.mount('/content/drive')
d = '/content/drive/MyDrive/Santa/'; os.makedirs(d, exist_ok=True); o = '/content/Santa/out/'
for a, b in [(o + 'santa.mp4', 'santa_impasto.mp4'), (o + 'santa_silent.mp4', 'santa_impasto_silent.mp4'), (o + 'check/sheet.jpg', 'sheet_impasto.jpg'), ('/content/run.log', 'run_impasto.log')]:
    shutil.copy(a, d + b)
print(sorted(os.listdir(d)))
drive.flush_and_unmount()